# Análise Exploratória de Dados — Dataset Titanic
Este notebook tem como objetivo realizar uma Análise Exploratória de Dados (EDA) completa sobre o clássico dataset do Titanic, desenvolvendo habilidades fundamentais para qualquer projeto de Machine Learning: entender, limpar, visualizar e interpretar os dados antes de construir modelos preditivos.

In [66]:
!pip install tensorflow numpy matplotlib seaborn scikit-learn

In [67]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import pandas as pd

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [68]:
# Carregando o dataset original do kaggle
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Carregando o dataset completo para exercícios iniciais
ds = pd.read_csv("titanic.csv")

#Parte 1 — Inspeção Inicial

##Objetivo: Entender a estrutura dos dados.

O dataset do Titanic contém informações sobre os passageiros a bordo, incluindo características demográficas, socioeconômicas e o resultado da tragédia (sobreviveu ou não). Antes de qualquer análise ou modelagem, é essencial inspecionar os dados brutos para compreender o que temos.
Enunciado:

1. Carregue o dataset titanic.csv com Pandas.  
2. Exiba o número de linhas e colunas.  
3. Liste os tipos de dados de cada coluna.  
4. Mostre as primeiras 10 linhas para inspeção visual.  

In [69]:
print(f"Dimensões do dataset: {ds.shape} Linhas x Colunas")

print(f"\nTipos de dados de cada coluna:\n{(ds.dtypes)}\n")

ds.head(10)

Dimensões do dataset: (891, 12) Linhas x Colunas

Tipos de dados de cada coluna:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


#Parte 2 — Tipos de Variáveis

##Objetivo: Classificar corretamente as variáveis.

Nem toda coluna de um dataset é igualmente útil para um modelo de predição. Identificar o tipo de cada variável — se é numérica ou categórica, discreta ou contínua, nominal ou ordinal — é um passo crucial para decidir como tratá-la nas etapas seguintes.
Enunciado:

Classifique cada coluna como numérica ou categórica.
Diga qual tipo de variável (qualitativa/quantitativa) e qual subtipo (discreta/contínua, nominal/ordinal).
Identifique colunas que dificilmente devem ser usadas em modelos de predição.

In [70]:
numeric_columns = ds.select_dtypes(include=['number'])
categoric_columns = ds.select_dtypes(include=['object'])

print(f"Colunas numéricas:\n{numeric_columns}")

print(f"\nColunas categóricas:\n{categoric_columns}")


Colunas numéricas:
     PassengerId  Survived  Pclass   Age  SibSp  Parch     Fare
0              1         0       3  22.0      1      0   7.2500
1              2         1       1  38.0      1      0  71.2833
2              3         1       3  26.0      0      0   7.9250
3              4         1       1  35.0      1      0  53.1000
4              5         0       3  35.0      0      0   8.0500
..           ...       ...     ...   ...    ...    ...      ...
886          887         0       2  27.0      0      0  13.0000
887          888         1       1  19.0      0      0  30.0000
888          889         0       3   NaN      1      2  23.4500
889          890         1       1  26.0      0      0  30.0000
890          891         0       3  32.0      0      0   7.7500

[891 rows x 7 columns]

Colunas categóricas:
                                                  Name     Sex  \
0                              Braund, Mr. Owen Harris    male   
1    Cumings, Mrs. John Bradley (Fl

#Parte 3 — Valores Faltantes
##Objetivo: Identificar e tratar valores ausentes.
Dados reais raramente são completos. Valores faltantes podem distorcer análises e prejudicar o treinamento de modelos. Por isso, é necessário identificá-los e decidir a melhor estratégia de imputação para cada caso.
Enunciado:

Verifique quantos valores faltantes há em cada coluna.
Para as variáveis numéricas com valores faltantes (e.g., Age), preencha com a mediana.
Para variáveis categóricas com valores faltantes (e.g., Cabin, Embarked), preencha com a moda.
Relate como isso alterou o número de valores faltantes.

In [71]:
# Tratamento de valores nulos
print(f"Quantidade de valores nulos por coluna: {ds.isna().sum()}")

Quantidade de valores nulos por coluna: PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


Para o tratamento das idades, para que este fique mais próximo do real,
foi observado que os nomes dos passageiros contém títulos em sua maioria.
Com isto, será feito uma mediana das idades dos passageiros com cada título,
para uma melhor aproximação das idades.

In [72]:
train_df['Title'] = train_df['Name'].str.extract(r',\s*([^\.]*)\.')
test_df['Title'] = test_df['Name'].str.extract(r',\s*([^\.]*)\.')

# Retornando uma series com os mesmos indices do orignal, aplicando a mediana para cada idade agrupada pelo titulo
# Obs: será retornado a mediana em todas as linhas, dependendo apenas do titulo, e não se o valor está vazio ou nao
age_by_title_train = train_df.groupby('Title')['Age'].transform('median')
age_by_title_test = test_df.groupby('Title')['Age'].transform('median')

# Preenchendo agora os valores nulos por indice
train_df['Age'].fillna(age_by_title_train, inplace=True)
test_df['Age'].fillna(age_by_title_test, inplace=True)

/tmp/ipykernel_4339/3370878297.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['Age'].fillna(age_by_title_train, inplace=True)
/tmp/ipykernel_4339/3370878297.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tr

In [73]:
# Preenchimento dos valores categóricos teste e treino
train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)
test_df['Embarked'].fillna(test_df['Embarked'].mode()[0], inplace=True)

train_df['Cabin'].fillna(train_df['Cabin'].mode()[0], inplace=True)
test_df['Cabin'].fillna(test_df['Cabin'].mode()[0], inplace=True)

display(train_df.isna().sum())
display(test_df.isna().sum())

# Preenchendo o Valor de Fare


/tmp/ipykernel_4339/2376939239.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['Embarked'].fillna(train_df['Embarked'].mode()[0], inplace=True)
/tmp/ipykernel_4339/2376939239.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(va

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


,0
PassengerId,0
Pclass,0
Name,0
Sex,0
Age,1
SibSp,0
Parch,0
Ticket,0
Fare,1
Cabin,0



#Parte 4 — Análise de Outliers
# Objetivo: Detectar outliers em variáveis contínuas.
Outliers são valores que se distanciam significativamente do padrão geral dos dados. Eles podem ser erros de medição, mas também podem representar casos reais e válidos — e a distinção entre os dois é fundamental.
Enunciado:

Escolha Age e Fare para análise.
Faça um boxplot para cada uma.
Calcule o IQR e filtre outliers (pontos além de 1.5 × IQR).
Quantos outliers foram identificados?
Discuta se esses valores podem ser erros ou casos reais de exceções válidas.

# Parte 5 — Distribuições e Histogramas
## Objetivo: Visualizar distribuições de variáveis.
Histogramas nos permitem enxergar como os valores de uma variável estão distribuídos — se são simétricos, assimétricos, concentrados em faixas específicas etc. Essa compreensão guia decisões de pré-processamento.
Enunciado:

Plote histogramas de Age e Fare.
Descreva a distribuição de idade (alguma assimetria?).
Compare o padrão de distribuição das tarifas por classe de passagem (Pclass).

# Parte 6 — Correlação entre Variáveis
## Objetivo: Analisar relações entre atributos.
A correlação mede o grau de relação linear (Pearson) ou monotônica (Spearman) entre pares de variáveis. Identificar correlações altas entre variáveis independentes pode indicar multicolinearidade — um problema para certos modelos.
Enunciado:

Calcule as matrizes de correlação de Pearson e de Spearman apenas entre variáveis numéricas.
Plote heatmaps das matrizes de correlação.
Identifique quais pares de variáveis apresentam alta correlação entre si. Há risco de multicolinearidade?
Identifique quais variáveis têm correlação mais forte com Survived.

# Parte 7 — Análise de Grupos
## Objetivo: Comparar grupos nos dados.
Além das correlações numéricas, é importante entender como variáveis categóricas se relacionam com a variável-alvo. No Titanic, fatores como sexo e classe social tiveram um papel histórico bem documentado na sobrevivência.
Enunciado:

Calcule a taxa de sobrevivência por:

Sexo (male, female)
Classe (Pclass)


Plote gráficos de barras que mostrem essas comparações.
Comente quais grupos tiveram maior taxa de sobrevivência e quais foram os possíveis motivos.

# Parte 8 — Scatter Plot Bivariado
## Objetivo: Explorar as relações entre duas variáveis.
Gráficos de dispersão permitem visualizar simultaneamente a relação entre duas variáveis numéricas e, ao adicionar cor, uma terceira dimensão — como a variável-alvo.
Enunciado:

Crie um gráfico de dispersão de Age vs Fare.
Colore os pontos de acordo com a sobrevivência (cores diferentes para Survived).
O gráfico mostra algum padrão entre idade, tarifa e sobrevivência?

#Parte 9 — Desafio: Engenharia de Atributos (Opcional)
## Objetivo: Criar novas features para enriquecer a análise e verificar, por meio de visualizações, se essas variáveis ajudam a explicar as diferenças nas taxas de sobrevivência.
Engenharia de atributos é o processo de criar novas variáveis a partir das existentes, com o objetivo de fornecer ao modelo informações mais ricas e relevantes que os dados brutos não expressam diretamente.
Enunciado:

Crie novas features como:

Tamanho da família: FamilySize = SibSp + Parch + 1
Título extraído do nome: Mr, Mrs, Miss, etc.


Avalie pelo menos uma nova visualização usando a nova variável.
Discuta como essa feature pode ajudar a explicar a diferença na sobrevivência.

# Parte Extra — Modelagem e Classificação
(A ser desenvolvida após a conclusão da EDA)
Com base em tudo que foi analisado nas seções anteriores, esta seção será dedicada à construção e avaliação de modelos de classificação para prever a sobrevivência dos passageiros.

# Conclusões
(A ser desenvolvida após todas as etapas intermediárias)